# 03 · So sánh Eclat Set và BitVector

**Mục đích:** khai thác cùng bộ S0 ở sáu ngưỡng support, kiểm tra exact parity và tự động chọn 3–4 mẫu đại diện để diễn giải.

**Điều kiện chạy:** Notebook 02 đã tạo TC7 manifest; Linux có systemd user manager/cgroup; Python 3.10+; Julia 1.13.

**Đầu vào:** `transactions/transactions.jsonl` và hash từ `tc7_manifest.toml`.

**Đầu ra:** 12 run status, `mining/parity_audit.json`, `mining/comparison.csv`, catalog ứng viên có đánh dấu kết quả tự chọn, `mining/representative_patterns.csv`, `mining/selection_manifest.toml` và `mining/selected_itemsets.csv` cho Notebook 04.

## 1. Tìm lượt TC7 và khởi tạo đường dẫn

Cell này chọn đúng một run có `tc7_manifest.toml` hoặc dùng `FASHION_RELEASE_RUN_ID`. Kiểm tra run root và đường dẫn transaction trước khi bật worker.


In [1]:
using Pkg
using SHA: sha256
using TOML

function find_repo_root(start=pwd())
    current = realpath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src", "EclatFIM.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("Không tìm thấy repository root")
        current = parent
    end
end

function find_tc7_root(repo_root)
    base = joinpath(repo_root, "results", "fashion", "release")
    isdir(base) || error("Chạy Notebook 01 và 02 trước")
    candidates = [joinpath(path, "transactions", "tc7_manifest.toml")
        for path in readdir(base; join=true)
        if isfile(joinpath(path, "transactions", "tc7_manifest.toml"))]
    requested = get(ENV, "FASHION_RELEASE_RUN_ID", "")
    if !isempty(requested)
        selected = joinpath(base, requested, "transactions", "tc7_manifest.toml")
        isfile(selected) || error("Không tìm thấy TC7 manifest cho $requested")
        return dirname(dirname(selected))
    end
    length(candidates) == 1 || error("Cần đúng một TC7 run; đặt FASHION_RELEASE_RUN_ID nếu có nhiều lượt")
    return dirname(dirname(only(candidates)))
end

repo_root = get(ENV, "FASHION_MNIST_REPO_ROOT", find_repo_root())
Pkg.activate(joinpath(repo_root, "environments", "fashion_notebooks"))
using EclatFIM
run_root = find_tc7_root(repo_root)
tc7_manifest_path = joinpath(run_root, "transactions", "tc7_manifest.toml")
tc7_manifest = TOML.parsefile(tc7_manifest_path)
transaction_path = joinpath(repo_root, tc7_manifest["artifacts"]["transactions"]["path"])
transaction_hash = bytes2hex(open(sha256, transaction_path))
@assert transaction_hash == tc7_manifest["transaction_sha256"]
@assert tc7_manifest["configuration"]["codebook"] == "B0"
@assert tc7_manifest["configuration"]["K"] == 32
@assert tc7_manifest["configuration"]["seed"] == 123
@assert tc7_manifest["configuration"]["transaction_variant"] == "S0"
@assert tc7_manifest["transaction_count"] == 70_000
println("Run: ", tc7_manifest["run_id"], " | transaction SHA-256: ", transaction_hash)
println("TC7 status: ", tc7_manifest["status"], " | transactions: ", tc7_manifest["transaction_count"])


  Activating project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/environments/fashion_notebooks`


Run: fashion_47265722452561 | transaction SHA-256: 4703e02391b753949c135ba4edd862465baa93a8565ec94385c0c34a70771d3b
TC7 status: PASS | transactions: 70000


## 2. Kiểm tra Linux/systemd và chạy smoke

Cell này kiểm tra Linux, xác minh user-systemd phản hồi, rồi chạy smoke cho worker, API, giới hạn tài nguyên và recovery. Chỉ tiếp tục khi smoke kết thúc với `p1_smoke: PASS`.


In [4]:
@assert Sys.islinux() "Mining release yêu cầu Linux user systemd"
python_candidates = filter(isfile, [
    something(Sys.which("python3"), ""),
    something(Sys.which("python"), ""),
    joinpath(repo_root, ".venv", "bin", "python3"),
    joinpath(repo_root, ".venv", "bin", "python")])
isempty(python_candidates) && error("Không tìm thấy Python 3 trong PATH hoặc .venv/bin")
PYTHON = first(python_candidates)
JULIA = joinpath(Sys.BINDIR, Base.julia_exename())
SMOKE = joinpath(repo_root, "scripts", "fashion_fim", "smoke.py")
println("Python: ", PYTHON)
run(Cmd([PYTHON, SMOKE, repo_root, JULIA]))
println("P1 smoke và systemd budget probe PASS")

Python: /usr/bin/python3
{
  "analysis": "PASS",
  "completed_runs": 3,
  "unique_patterns": 3,
  "representatives": 3,
  "threshold": 0.9,
  "relaxed": true,
  "files": [
    "analysis/count_by_length.csv",
    "analysis/pattern_support_summary.csv",
    "analysis/master_patterns.jsonl",
    "analysis/run_statistics.csv",
    "analysis/pattern_statistics.md",
    "analysis/representative_patterns.csv",
    "analysis/minsup_vs_itemset_count.svg",
    "analysis/minsup_vs_runtime.svg",
    "analysis/itemset_length_distribution.svg",
    "figures/minsup_vs_itemset_count.png",
    "figures/minsup_vs_runtime.png",
    "figures/itemset_lengths_by_minsup.png"
  ]
}
PASS Julia loader keeps row/TID/image mapping
PASS public Eclat API returns expected Dict and optional-limit semantics
PASS toy full JSONL output is sorted and support fractions are transaction-based
PASS COMPLETED_ZERO writes a 0-byte exact staging file
PASS output_limit_bytes stops before overflow
PASS systemd user service enforc

## 3. Chuẩn bị runtime manifest dùng chung

Cell này ghi Julia/Python executable, versions, threads, depot/load paths và process environment. Hai backend nhận cùng runtime và cùng giới hạn 1 thread, 3.072 MiB RAM, 3.600 giây/run, 8 GiB/output và không swap.


In [5]:
runtime_env_path = joinpath(run_root, "mining", "runtime_env.toml")
mkpath(dirname(runtime_env_path))
runtime_env = Dict{String,Any}(
    "julia_executable" => JULIA,
    "julia_version" => string(VERSION),
    "julia_threads" => 1,
    "julia_depot_path" => join(DEPOT_PATH, ":"),
    "julia_load_path" => join(LOAD_PATH, ":"),
    "home_path" => homedir(),
    "process_path" => get(ENV, "PATH", "/usr/bin:/bin"),
    "cpu_model" => get(ENV, "PROCESSOR_IDENTIFIER", "Linux CPU"),
    "memory_available_bytes" => Sys.free_memory(),
    "julia_package_versions" => String[])
open(runtime_env_path, "w") do io
    TOML.print(io, runtime_env)
end
SUPERVISOR = joinpath(repo_root, "scripts", "fashion_fim", "supervisor.py")
AUDITOR = joinpath(repo_root, "scripts", "fashion_fim", "audit_outputs.py")
SELECTOR = joinpath(repo_root, "scripts", "fashion_fim", "selection.py")
SET_OUTPUT_DIR = joinpath(run_root, "mining", "backends", "set")
BITVECTOR_OUTPUT_DIR = joinpath(run_root, "mining", "backends", "bitvector")
println("Runtime manifest: ", relpath(runtime_env_path, repo_root))


Runtime manifest: results/fashion/release/fashion_47265722452561/mining/runtime_env.toml


## 4. Khởi tạo hai backend với cùng transaction hash và resource limits

Cell này tạo sáu threshold sidecar riêng cho Set và BitVector. Mỗi backend có namespace riêng; in ra input hash và status ban đầu để kiểm tra rằng hai lượt dùng cùng S0.


In [6]:
function initialize_backend(backend, output_dir)
    run(Cmd([PYTHON, SUPERVISOR, "--init",
        "--repo-root", repo_root,
        "--input-path", transaction_path,
        "--output-dir", output_dir,
        "--runtime-env", runtime_env_path,
        "--backend", backend,
        "--wall-time-seconds-per-run", "3600",
        "--memory-limit-mb-per-worker", "3072",
        "--output-limit-bytes", string(8 * 1024^3),
        "--budget-revision", "release-b1"]))
end
initialize_backend("set", SET_OUTPUT_DIR)
initialize_backend("bitvector", BITVECTOR_OUTPUT_DIR)
println("Initialized Set and BitVector namespaces for ", transaction_hash)


{
  "initialized": true,
  "output_dir": "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561/mining/backends/set",
  "input_sha256": "4703e02391b753949c135ba4edd862465baa93a8565ec94385c0c34a70771d3b",
  "guardrails": {
    "max_singleton": 48922,
    "support_6": 43263,
    "support_26": 48922,
    "support_6_26": 32266
  },
  "statuses": [
    {
      "run_id": "set_minsup_80_4703e023_2a3f35c1_release-b1_a01",
      "configuration": "B0_K32_seed123_S0",
      "backend": "set",
      "minsup_fraction": "0.8",
      "minsup_count": "56000",
      "status": "NOT_RUN",
      "reason_code": "",
      "total_frequent_itemsets": "NA",
      "max_itemset_length": "NA",
      "observed_partial_itemsets": "0",
      "runtime_seconds": "NA",
      "peak_rss_mb": "NA",
      "output_size_bytes": "NA",
      "engine_revision": "ef1809a2816a345c2db19741c06357c7fbee4572",
      "input_sha256": "4703e02391b75394

## 5. Chạy sáu ngưỡng bằng Set

Cell này chạy Set tuần tự ở 80%, 60%, 40%, 20%, 10%, 5%. Worker dùng systemd với resource limits đã ghi; kiểm tra sáu trạng thái cuối trong log/CSV.


In [7]:
run(Cmd([PYTHON, SUPERVISOR, "--run", "--output-dir", SET_OUTPUT_DIR]))
println("Set batch finished.")


COMPLETED_ZERO 80% count=0 runtime=6.13114627199684
COMPLETED_FULL 60% count=2 runtime=6.102797616003954
COMPLETED_FULL 40% count=15 runtime=6.075652021005226
COMPLETED_FULL 20% count=174 runtime=6.119978734001052
COMPLETED_FULL 10% count=1275 runtime=9.141710653995688
COMPLETED_FULL 5% count=7228 runtime=13.173820010997588
[
  {
    "run_id": "set_minsup_80_4703e023_2a3f35c1_release-b1_a01",
    "configuration": "B0_K32_seed123_S0",
    "backend": "set",
    "minsup_fraction": "0.8",
    "minsup_count": "56000",
    "status": "COMPLETED_ZERO",
    "reason_code": "",
    "total_frequent_itemsets": "0",
    "max_itemset_length": "0",
    "observed_partial_itemsets": "0",
    "runtime_seconds": "6.13114627199684",
    "peak_rss_mb": "436.14453125",
    "output_size_bytes": "0",
    "engine_revision": "ef1809a2816a345c2db19741c06357c7fbee4572",
    "input_sha256": "4703e02391b753949c135ba4edd862465baa93a8565ec94385c0c34a70771d3b",
    "used_optional_limits": "False",
    "worker_log_path"

## 6. Chạy sáu ngưỡng bằng BitVector

Cell này chạy BitVector sau khi batch Set kết thúc, cùng transaction, thresholds, Julia threads và giới hạn tài nguyên. Kiểm tra cả sáu trạng thái cuối.


In [8]:
run(Cmd([PYTHON, SUPERVISOR, "--run", "--output-dir", BITVECTOR_OUTPUT_DIR]))
println("BitVector batch finished.")


COMPLETED_ZERO 80% count=0 runtime=8.17828136600292
COMPLETED_FULL 60% count=2 runtime=7.145373938998091
COMPLETED_FULL 40% count=15 runtime=6.084185873005481
COMPLETED_FULL 20% count=174 runtime=6.097838043999218
COMPLETED_FULL 10% count=1275 runtime=8.140609250003763
COMPLETED_FULL 5% count=7228 runtime=8.195575247002125
[
  {
    "run_id": "bitvector_minsup_80_4703e023_2a3f35c1_release-b1_a01",
    "configuration": "B0_K32_seed123_S0",
    "backend": "bitvector",
    "minsup_fraction": "0.8",
    "minsup_count": "56000",
    "status": "COMPLETED_ZERO",
    "reason_code": "",
    "total_frequent_itemsets": "0",
    "max_itemset_length": "0",
    "observed_partial_itemsets": "0",
    "runtime_seconds": "8.17828136600292",
    "peak_rss_mb": "445.515625",
    "output_size_bytes": "0",
    "engine_revision": "ef1809a2816a345c2db19741c06357c7fbee4572",
    "input_sha256": "4703e02391b753949c135ba4edd862465baa93a8565ec94385c0c34a70771d3b",
    "used_optional_limits": "False",
    "worker_

## 7. Hậu kiểm support và exact parity

Cell này duyệt mọi itemset trong cả 12 output, tính lại support từ 70.000 transaction bằng vertical bit masks và so sánh chính xác key/count của từng cặp. Mở `parity_audit.json` và kiểm tra 12 dòng `comparison.csv`; historical 7.228 chỉ áp dụng khi input hash trùng nguồn cũ.


In [9]:
run(Cmd([PYTHON, AUDITOR,
    "--release-root", run_root,
    "--set-output-dir", SET_OUTPUT_DIR,
    "--bitvector-output-dir", BITVECTOR_OUTPUT_DIR,
    "--input-path", transaction_path]))
parity_path = joinpath(run_root, "mining", "parity_audit.json")
parity_text = read(parity_path, String)
@assert occursin("exact_set_bitvector_release_parity", parity_text)
println(parity_text[1:min(end, 5000)])


{
  "status": "PASS",
  "transaction_count": 70000,
  "thresholds": 6,
  "historical_7228_reference": "NOT_APPLICABLE",
  "artifact": "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561/mining/parity_audit.json"
}
{
  "schema_version": 1,
  "audit_type": "exact_set_bitvector_release_parity",
  "status": "PASS",
  "input_sha256": "4703e02391b753949c135ba4edd862465baa93a8565ec94385c0c34a70771d3b",
  "transaction_count": 70000,
  "historical_7228_reference": "NOT_APPLICABLE",
  "direct_support_method": "Vertical transaction bit masks; support is the population count of item-mask intersections",
  "thresholds": [
    {
      "min_count": 56000,
      "minsup_percent": 80,
      "set_status": "COMPLETED_ZERO",
      "bitvector_status": "COMPLETED_ZERO",
      "parity": {
        "status": "PASS",
        "missing_count": 0,
        "extra_count": 0,
        "support_mismatch_count": 0,
        "missing

## 8. Tự động chọn 3–4 mẫu đại diện

Cell này hợp nhất itemset Set ở cả sáu ngưỡng, rồi xác minh trạng thái và SHA-256 của từng đầu ra. Selector chia mẫu theo độ dài 1, 2, ≥3 và tertile support tính trên các support count phân biệt; các mẫu đồng support luôn cùng nhóm. Để mẫu nhỏ vẫn phủ đủ độ dài, selector lấy trước lần lượt một mẫu từ các ô (1/high), (2/mid), (≥3/low). Trong mỗi ô, thứ tự ưu tiên là support giảm dần, độ dài tăng dần rồi dãy visual-word ID theo thứ tự từ điển. Mẫu được nhận khi Jaccard với từng mẫu đã chọn đều ≤0,80. Sau đó selector lấp chỗ còn lại bằng cách đi vòng cố định qua 9 ô độ dài/support. Nếu chưa đủ 4 mẫu, selector chạy lại toàn bộ một lần với Jaccard ≤0,90; cần ít nhất 3 mẫu, nếu không sẽ báo lỗi. Kiểm tra `selection_manifest.toml`, `representative_patterns.csv` và `selected_itemsets.csv`; Notebook 04 đọc selection CSV đã lưu, không cần chọn tay.

In [ ]:
python_candidates = filter(isfile, [
    something(Sys.which("python3"), ""),
    something(Sys.which("python"), ""),
    joinpath(repo_root, ".venv", "bin", "python3"),
    joinpath(repo_root, ".venv", "bin", "python")])
isempty(python_candidates) && error("Không tìm thấy Python 3 trong PATH hoặc .venv/bin")
python_executable = first(python_candidates)
selector_path = joinpath(repo_root, "scripts", "fashion_fim", "selection.py")
set_output_dir = joinpath(run_root, "mining", "backends", "set")
run(Cmd([python_executable, selector_path,
    "--release-root", run_root,
    "--set-output-dir", set_output_dir]))
selection_manifest_path = joinpath(run_root, "mining", "selection_manifest.toml")
selected_path = joinpath(run_root, "mining", "selected_itemsets.csv")
representative_path = joinpath(run_root, "mining", "representative_patterns.csv")
@assert isfile(selection_manifest_path) && isfile(selected_path) && isfile(representative_path)
selection_manifest = TOML.parsefile(selection_manifest_path)
@assert 3 <= selection_manifest["target_selected"] <= 4
@assert selection_manifest["unique_pattern_count"] > 0
println("Selection rule: ", selection_manifest["rule_version"])
println("Selected: ", selection_manifest["target_selected"], " / 4; Jaccard limit used: ",
        selection_manifest["actual_jaccard_threshold"],
        "; relaxed: ", selection_manifest["relaxation_used"])
println("Selected patterns:")
for line in Iterators.take(eachline(representative_path), 5)
    println(line)
end
println("Notebook 04 handoff: ", relpath(selected_path, repo_root))
println("Selection manifest: ", relpath(selection_manifest_path, repo_root))